# Reproducing Welch et al. (2025) from dandiset 001863

**Paper.** Welch AC, Gonzales-Hess N, Tarvin T, Connor S, Smear M (2025). The impact of a
psychedelic drug on olfactory search behavior by mice. bioRxiv
[doi:10.1101/2025.07.09.663970](https://doi.org/10.1101/2025.07.09.663970). The claims file was
written from the full text of the preprint, version 1 (its JATS XML), which has its supplemental
figure legends in the text and no separate supplement.

**Dandiset.** [DANDI:001863](https://dandiarchive.org/dandiset/001863/0.260806.2109), version
0.260806.2109. The paper cites no DANDI version (it predates the dandiset). This is the only
published version; the draft has the same 44 assets and size, so the published version is pinned.

**What the dandiset holds.** 44 NWB files, one DOI and one saline session for each of 22 mice:
the trial table, DeepLabCut tracking of 8 keypoints, the sniff cycles detected from the nasal
thermistor (31 sessions) and an unnamed event series that the authors' conversion script and
figure code identify as the head-twitch (HTR) times (32 sessions). The reward-port entries and
exits of Figure 2 are not in it.

**Outside inputs.** The authors' code, github.com/Smear-Lab/Olfactory-search-on-psychedelics at
commit aa80af0a153c6bf099e5668d27c9d88e2bfc033f, which the claims file cites for its method
details. It also names the event series as HTR times (claim `htr-event-link`).

| Claim | Tested here |
|---|---|
| fig1b-accuracy-lower-doi, fig1c-accuracy-time-bins, fig1d-fewer-trials, fig1d-fewer-trials-females-only, fig1e-repoke-no-difference, iti-false-report-no-difference | Tested |
| fig2a to fig2d and their supplements (9 claims) | Blocked: data missing (no reward-port entries or exits) |
| fig3a-odor-trial-duration-longer, fig3a-omission-duration-no-difference, fig3b-iti-duration-no-difference | Tested |
| fig3c, fig3d and the five speed claims of the Figure 3 supplement | Tested |
| fig4abc, fig4supp-ks-every-mouse, fig4d, fig4e | Tested |
| fig5ab-occupancy-similar | Tested |
| htr-event-link (added here) and the five HTR claims of Figure 5 | Tested |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import doi_search as ds

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
sessions = ds.load_all()
print(f"loaded {len(sessions)} files from dandiset {ds.DANDISET_ID} version {ds.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")
by_key = {(s["mouse"], s["condition"]): s for s in sessions}

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "mouse": s["mouse"], "session": s["session"], "condition": s["condition"],
    "drug_field": s["drug"], "sex_code": s["sex"], "sex_file": s["sex_file"], "cohort": s["cohort"],
    "trials": len(s["trials"]), "odor_trials": int(ds.odor_mask(s["trials"]).sum()),
    "trial_columns": len(s["trials"].columns), "frames": len(s["nose_xy"]),
    "file_rate": round(s["file_rate"], 2),
    "sniff_rows": 0 if s["sniff_start"] is None else len(s["sniff_start"]),
    "events": 0 if s["events"] is None else len(s["events"]),
    "event_source": None if s["events"] is None else s["events_source"],
    "start": s["start"][:10]} for s in sessions]).sort_values(["mouse", "condition"])
print(info.to_string(index=False))
print("\nSessions by condition:", info.condition.value_counts().to_dict(), "; mice:", info.mouse.nunique())
print("Every mouse has one DOI and one saline session, the code's session numbers:",
      bool((info.groupby("mouse").condition.apply(sorted).apply(tuple) == ("DOI", "saline")).all()))
print("Condition from the code's session list agrees with the file's drug field:",
      bool((info.condition.str.lower() == info.drug_field).all()))
print("Sex in the file differs from the code's list:",
      info[info.sex_code != info.sex_file][["mouse", "session", "condition", "sex_code", "sex_file"]].to_dict("records"))
print("Mice per sex (code):", info.drop_duplicates("mouse").sex_code.value_counts().to_dict())
print("Sessions with fewer than", ds.MIN_TRIALS, "trials:", int((info.trials < ds.MIN_TRIALS).sum()))

The 44 files are the 22 mice and the DOI and saline sessions of the code's list, and the file's
drug field agrees for every session. The paper's 12 females and 10 males are the code's list. One
file disagrees with it: the saline session of mouse 2182 says female while its DOI session says
male (the code and the claims file have 2182 as male). The analysis uses the code's list. No
session has fewer than 40 trials, so the code's 40-trial filter drops none. The first cohort's
trial tables have 6 columns and no `false_alarm` or `repoke` (Methods, Figure 1: "17 due to data
acquisition updates").

In [ ]:
tr_all = pd.concat([s["trials"].assign(mouse=s["mouse"], condition=s["condition"], cohort=s["cohort"])
                    for s in sessions])
print("odor_control values:", tr_all.odor_control.value_counts().to_dict())
print("outcome values:", tr_all.outcome.value_counts().to_dict(), "; side values:", tr_all.side.value_counts().to_dict())
print("Omission share of trials: %.3f (Methods: 10%%)" % (tr_all.odor_control == 3).mean())
print("Set ITI (s) range:", tr_all.iti_time.min(), "to", tr_all.iti_time.max(), "(Methods: 1 to 15 s)")
over = tr_all[tr_all.iti_time > 15]
print("Trials with a set ITI above 15 s:", len(over), "in sessions", sorted(set(zip(over.mouse, over.condition))),
      "; values", over.iti_time.value_counts().to_dict())
dur = tr_all.stop_time - tr_all.start_time
print("Trial duration (s): min %.2f, median %.2f, max %.1f" % (dur.min(), dur.median(), dur.max()))
print("Trial start (s): min %.1f, max %.1f; last trial end per session, min %.0f max %.0f"
      % (tr_all.start_time.min(), tr_all.start_time.max(),
         tr_all.groupby(["mouse", "condition"]).stop_time.max().min(), tr_all.groupby(["mouse", "condition"]).stop_time.max().max()))
iti_obs = np.concatenate([ds.iti_lengths(s["trials"])[:-1] for s in sessions])
print("Observed ITI, next start minus end (s): min %.2f, median %.2f, max %.1f; negative: %d"
      % (iti_obs.min(), np.median(iti_obs), iti_obs.max(), (iti_obs < 0).sum()))
later = tr_all[tr_all.cohort == "later"]
print("false_alarm values (later cohort):", later.false_alarm.value_counts().to_dict())
print("repoke values (later cohort):", later.repoke.value_counts().sort_index().to_dict())

# The session notes give, for the later cohort, correct over total trials by side.
rows = []
for s in sessions:
    n, t = s["notes"], s["trials"]
    if s["cohort"] != "later":
        continue
    def frac(k):
        a, b = n.get(k, "0/0").split("/")
        return int(a), int(b)
    o = ds.odor_mask(t)
    rows.append({"mouse": s["mouse"], "condition": s["condition"],
                 "notes_L": frac("L"), "notes_R": frac("R"),
                 "table_side1": (int(t.outcome[o & (t.side == 1)].sum()), int((o & (t.side == 1)).sum())),
                 "table_side2": (int(t.outcome[o & (t.side == 2)].sum()), int((o & (t.side == 2)).sum()))})
notes_check = pd.DataFrame(rows)
match1 = (notes_check.notes_L == notes_check.table_side1) & (notes_check.notes_R == notes_check.table_side2)
match2 = (notes_check.notes_L == notes_check.table_side2) & (notes_check.notes_R == notes_check.table_side1)
print("\nLater-cohort sessions whose notes' L and R (correct/total odor trials) equal the table's side 1 and 2:",
      int(match1.sum()), "; side 2 and 1:", int(match2.sum()), "; of", len(notes_check))
print(notes_check[~(match1 | match2)].to_string(index=False))

The trial table holds odor trials (0) and omission trials (3), about 10% omission as the Methods
say, set ITIs of 1 to 15 s, outcomes 0 and 1 and sides 1 and 2. The ITI observed between trials
is longer than the set ITI, as it includes the time to poke into the initiation port. In the five
first-cohort mice 347 trials have a set ITI of 16 to 25 s, beyond the Methods' 1 to 15 s; the
set ITI is not used by any claim. The session notes
count correct and total odor trials by side, and in all 34 later-cohort sessions they equal the
table's counts with side 2 as left and side 1 as right; the files do not say which side is which.
`false_alarm` is 1 in 6 of 3261 trials, as the paper says false reports were rare.

### Do the tracking frames share the trial table's clock?

The files give the tracking no timestamps: each series has a starting time of 0 and a rate that
the conversion script estimated as the number of frames over the last trial's end time (its
comments call it an approximation). The paper's code places frame i at i / 80 s. A trial starts
at the initiation port and ends when the mouse reaches the odor side at the far end of the arena,
so the nose's x at the end of a trial minus its x at the start (first cohort flipped) shows
whether frames and trials are aligned. A rate and offset are fitted for each session
(`ds.fit_clock`), and refitted on each half of the session.

In [ ]:
clock_rows = []
x_by_session = {}
for s in sessions:
    x, y, cinfo = ds.clean_nose(s)
    if s["cohort"] == "first":
        x = ds.FLIP_X - x
    fit = ds.fit_clock(x, s["trials"])
    shifted = s["trials"].assign(start_time=s["trials"].start_time + 30, stop_time=s["trials"].stop_time + 30)
    clock_rows.append({"mouse": s["mouse"], "condition": s["condition"], "cohort": s["cohort"],
                       "frames": len(x), "file_rate": round(s["file_rate"], 2),
                       "disp_code_80fps": ds.trial_displacement(x, s["trials"], ds.CODE_FPS, 0.0),
                       "disp_file_rate": ds.trial_displacement(x, s["trials"], s["file_rate"], 0.0),
                       "fit_fps": fit["fps"], "fit_offset_s": fit["offset"], "disp_fit": fit["displacement"],
                       "disp_fit_trials_shifted_30s": ds.trial_displacement(x, shifted, fit["fps"], fit["offset"]),
                       "offset_1st_half": fit["offset_first_half"], "offset_2nd_half": fit["offset_second_half"],
                       "nose_low_likelihood": cinfo["low_likelihood"] / cinfo["frames"],
                       "nose_interpolated": cinfo["interpolated"] / cinfo["frames"]})
clock = pd.DataFrame(clock_rows).sort_values(["cohort", "mouse", "condition"])
print(clock.round(3).to_string(index=False))
print("\nMedian displacement (px) by cohort:")
print(clock.groupby("cohort")[["disp_code_80fps", "disp_file_rate", "disp_fit", "disp_fit_trials_shifted_30s"]].median().round(1).to_string())
print("\nFitted rate by cohort (frames/s):", clock.groupby("cohort").fit_fps.describe()[["min", "50%", "max"]].round(2).to_dict("index"))
clock["halves_agree"] = (clock.offset_1st_half - clock.offset_2nd_half).abs() <= 1.0
clock["fit_ok"] = (clock.disp_fit >= 100) & clock.halves_agree
print("Sessions whose fit is weak (displacement < 100 px) or whose halves differ by more than 1 s:")
print(clock[~clock.fit_ok][["mouse", "condition", "fit_fps", "fit_offset_s", "disp_fit", "offset_1st_half", "offset_2nd_half"]].to_string(index=False))
print("\nFraction of nose frames below likelihood 0.95: median %.2f, range %.2f to %.2f"
      % (clock.nose_low_likelihood.median(), clock.nose_low_likelihood.min(), clock.nose_low_likelihood.max()))
CLOCK = {(r.mouse, r.condition): (r.fit_fps, r.fit_offset_s) for r in clock.itertuples()}

With the code's clock (frame i at i / 80 s) the nose moves by a median of about 0 px between the
start and the end of a trial: the frames the code labels as within a trial are not the trial's
frames. The file's own rate does no better. A linear clock fitted to each session moves the nose
by hundreds of pixels toward the odor end over a trial, and the same offset fits both halves of
most sessions. The fitted rate is near 80.5 frames/s in the later cohort (the 80 frames/s of the
code drifts by about 0.5 x 2400 / 80, some 15 s, over a session, more than a trial lasts) and 72
to 75 frames/s in the first cohort, whose videos hold fewer frames than 40 min at 80 frames/s. The
sessions printed as weak are not established. Shifting the trials by 30 s under the fitted clock
removes the displacement, so the fit is a response to the trials and not an offset.

This is a finding about the dataset: the tracking carries no frame times and the stated rate is an
estimate that does not align it with the trials. The claims file fixes the code's clock for the
analyses that split frames by trial period (speed, HTR maps); they are run with it, as the paper
did, and again with the fitted clock as a variant. The nose keypoint is below the 0.95 likelihood
cut in most frames of many sessions, so most nose positions are interpolated, as the code does.

In [ ]:
# Sniff clock: sniff frequency is higher within trials than in the ITI; compare with the trial
# labels shifted in time.
shifts = [-30, -10, -5, -2, 0, 2, 5, 10, 30]
rows = []
for s in sessions:
    d = ds.sniff_table(s)
    if d is None:
        continue
    row = {"mouse": s["mouse"], "condition": s["condition"], "rows": len(s["sniff_start"]),
           "faulty_in_code": (s["mouse"], s["session"]) in ds.FAULTY_SNIFF,
           "first_s": d.t.min(), "last_s": d.t.max(),
           "inh_to_exh_ms_median": 1000 * np.median(s["sniff_stop"] - s["sniff_start"])}
    for sh in shifts:
        lab = ds.label_periods(d.t.to_numpy(), s["trials"].assign(start_time=s["trials"].start_time + sh,
                                                                  stop_time=s["trials"].stop_time + sh))
        row[f"shift_{sh}"] = np.median(d.frequency[lab > 0]) - np.median(d.frequency[lab == 0])
    rows.append(row)
sniff_clock = pd.DataFrame(rows).sort_values(["mouse", "condition"])
print(sniff_clock.round(2).to_string(index=False))
good = sniff_clock[~sniff_clock.faulty_in_code]
print("\nUsable sessions: within-trial minus ITI median frequency (Hz), median over sessions, by shift of the trial labels (s):")
print(good[[f"shift_{sh}" for sh in shifts]].median().round(3).to_string())
print("Sessions where the true labels give a larger contrast than every shift of 10 s or more:",
      int((good.shift_0 > good[["shift_-30", "shift_-10", "shift_10", "shift_30"]].max(axis=1)).sum()), "of", len(good))
print("\nSessions with sniff cycles by condition (usable, code's exclusions):",
      good.condition.value_counts().to_dict(), "; all:", sniff_clock.condition.value_counts().to_dict())

The sniff cycles span the 40 min session (0 to about 2400 s), and sniffing is faster within
trials than in the ITI with the true trial labels, more than with labels shifted by 10 s or more
in most usable sessions; the contrast peaks between 0 and 2 s of shift. This is consistent with a
shared clock to within a few seconds, not a proof of it. The code's four faulty sessions keep only
a few hundred to a few thousand rows of tens of thousands (the files' descriptions give the rows
dropped for a missing exhalation). Without them, 14 DOI and 13 saline sessions remain.

In [ ]:
# The event series: values, counts by condition, and the form of the timestamps.
ev = info[info.events > 0]
print("Data values of the series (all 1):", sorted({float(v) for s in sessions if s["events"] is not None for v in s["events_data"]}))
print("Sessions with events by condition:", ev.condition.value_counts().to_dict())
print("Events per session, DOI: median %d (range %d to %d); saline: median %d (range %d to %d)" % (
    ev[ev.condition == "DOI"].events.median(), ev[ev.condition == "DOI"].events.min(), ev[ev.condition == "DOI"].events.max(),
    ev[ev.condition == "saline"].events.median(), ev[ev.condition == "saline"].events.min(), ev[ev.condition == "saline"].events.max()))
frac_rows = []
for s in sessions:
    if s["events"] is None:
        continue
    t = s["events"]
    frac_rows.append({"mouse": s["mouse"], "condition": s["condition"], "source": s["events_source"],
                      "max_last_field_x100": int(np.round((t - np.floor(t + 1e-9)) * 100).max()),
                      "hundredths_only": bool(np.allclose(np.round(t * 100), t * 100, atol=1e-6)),
                      "first_s": t.min(), "last_s": t.max()})
fr = pd.DataFrame(frac_rows)
print(fr.groupby("source").agg(sessions=("mouse", "size"), max_field=("max_last_field_x100", "max"),
                               hundredths_only=("hundredths_only", "all"), last_s=("last_s", "max")).to_string())

The series has no name or unit beyond "Event times (s) from Time Stamps Task", and its values are
all 1. It is in all 22 DOI sessions (24 to 204 events) and in 10 saline sessions (14 to 35). In
the later cohort the last field of the stored timestamps, read as hundredths by the
conversion script, never exceeds 79: it is a frame count at 80 frames/s (a video timecode), so
the stored times are off by up to 0.2 s, and the code, which reads the field as thousandths, by up
to 0.7 s. The first cohort's manually scored files are in seconds, in steps of 5 ms. The HTR
claims are run under the code's reading and shown under the other two.

In [ ]:
# Recordings deposited twice: compare trial tables and nose tracks between every pair of sessions.
sig = []
for s in sessions:
    sig.append(((s["mouse"], s["condition"]), len(s["trials"]), s["trials"].start_time.to_numpy()[:20].round(2),
                np.nan_to_num(s["nose_xy"][:2000, 0]).round(1), len(s["nose_xy"])))
pairs = []
for i in range(len(sig)):
    for j in range(i + 1, len(sig)):
        same_trials = sig[i][1] == sig[j][1] and np.array_equal(sig[i][2], sig[j][2])
        same_track = sig[i][4] == sig[j][4] and np.array_equal(sig[i][3], sig[j][3])
        if same_trials or same_track:
            pairs.append((sig[i][0], sig[j][0], same_trials, same_track))
print("Pairs of sessions with identical trial starts or identical nose tracks:", pairs if pairs else "none")
print("Distinct session start times:", info.start.nunique(), "dates for", len(info), "sessions")

No session's trial table or nose track repeats another's, so each recording is counted once.
Each mouse is the same animal in its two sessions by its subject id, as the paired design and the
code have it.

## Figure 1: accuracy and task engagement

Per session from the trial table: accuracy on odor trials (percent correct), the trial counts,
and for the later cohort the repoke rate and false reports (fig1_cleaned.Rmd, as the claims file
gives it).

In [ ]:
rows = []
for s in sessions:
    t = s["trials"]
    o = ds.odor_mask(t)
    r = {"mouse": s["mouse"], "condition": s["condition"], "sex": s["sex"], "cohort": s["cohort"],
         "trials_all": len(t), "trials_odor": int(o.sum()),
         "acc_odor": 100 * t.outcome[o].mean(), "acc_omission": 100 * t.outcome[~o].mean(),
         "dur_odor": (t.stop_time - t.start_time)[o].mean(), "dur_omission": (t.stop_time - t.start_time)[~o].mean(),
         "iti": np.nanmean(ds.iti_lengths(t))}
    if "repoke" in t:
        r.update(repoke_rate=t.repoke.sum() / (len(t) - 1), false_reports=t.false_alarm.sum(),
                 false_report_rate=t.false_alarm.sum() / (len(t) - 1))
    rows.append(r)
per_session = pd.DataFrame(rows)
wide = per_session.pivot(index="mouse", columns="condition")
sex_of = per_session.drop_duplicates("mouse").set_index("mouse").sex


def paired(col, mice=None, label=None):
    w = wide if mice is None else wide.loc[mice]
    a, b = w[(col, "DOI")], w[(col, "saline")]
    ok = a.notna() & b.notna()
    v, p = ds.r_signrank(a[ok], b[ok])
    diff = a[ok] - b[ok]
    return {"measure": label or col, "n": int(ok.sum()), "DOI_median": a[ok].median(), "saline_median": b[ok].median(),
            "median_DOI_minus_saline": diff.median(), "DOI_lower_in": int((diff < 0).sum()), "V": v, "p": p}

### fig1b-accuracy-lower-doi

> Mice are less accurate in the olfactory search task in the DOI session than in the saline
> session, going more often to the incorrect side.

In [ ]:
res = pd.DataFrame([paired("acc_odor", label="odor-trial accuracy (%)"),
                    paired("acc_omission", label="omission-trial accuracy (%), variant")])
print(res.round(4).to_string(index=False))
print("\nBy sex, odor-trial accuracy:")
print(pd.DataFrame([paired("acc_odor", [m for m in wide.index if sex_of[m] == sx], f"sex {sx}") for sx in "FM"]).round(4).to_string(index=False))
print("\nPer mouse (odor-trial accuracy, %):")
print(wide["acc_odor"].assign(sex=sex_of).round(1).T.to_string())
fig, ax = plt.subplots(figsize=(4, 4))
for sx, c in (("F", "#990068"), ("M", "#00B19E")):
    m = sex_of[sex_of == sx].index
    ax.scatter(wide.loc[m, ("acc_odor", "saline")], wide.loc[m, ("acc_odor", "DOI")], c=c, label=sx)
ax.plot([60, 100], [60, 100], "k:")
ax.set(xlabel="saline accuracy (%)", ylabel="DOI accuracy (%)", xlim=(60, 100), ylim=(60, 100))
ax.legend()
plt.show()
print("Plotted range: saline %.1f to %.1f, DOI %.1f to %.1f; none outside the axes."
      % (wide[("acc_odor", "saline")].min(), wide[("acc_odor", "saline")].max(), wide[("acc_odor", "DOI")].min(), wide[("acc_odor", "DOI")].max()))

**Result: reproduced.** Over the 22 mice with paired sessions (the expected 22), odor-trial
accuracy is lower on DOI (median difference -3.2 points, lower in 14 of 22 mice), signed-rank
p = 0.023 against the paper's 0.024. Omission-trial accuracy does not differ significantly (variant).

### fig1c-accuracy-time-bins

> Over the session, odor-trial accuracy in 5 min bins is higher in the saline condition than the
> DOI condition after the first 10 min, except in the 30 to 35 min bin, so the effect is spread
> across the session.

In [ ]:
rows = []
for s in sessions:
    t = s["trials"]
    o = ds.odor_mask(t)
    b = pd.cut(t.start_time.to_numpy() / 60, bins=np.arange(0, 41, 5), include_lowest=True, labels=False)
    g = pd.DataFrame({"bin": b, "rolling": ds.rolling_accuracy(t), "success": t.outcome})[o].groupby("bin")
    for k, v in g.agg(rolling=("rolling", "mean"), fraction=("success", "mean")).iterrows():
        rows.append({"mouse": s["mouse"], "condition": s["condition"], "bin": int(k), "rolling": v["rolling"], "fraction": 100 * v["fraction"]})
binned = pd.DataFrame(rows)
out = []
for k, g in binned.groupby("bin"):
    rec = {"bin_min": f"{5 * k}-{5 * k + 5}"}
    for col in ("rolling", "fraction"):
        a, c = g[g.condition == "DOI"][col], g[g.condition == "saline"][col]
        w, p = ds.r_ranksum(a, c)
        rec.update({f"{col}_n_DOI": len(a), f"{col}_n_sal": len(c), f"{col}_DOI_mean": a.mean(),
                    f"{col}_saline_mean": c.mean(), f"{col}_saline_higher": c.mean() > a.mean(), f"{col}_p": p})
    out.append(rec)
bins_res = pd.DataFrame(out)
print(bins_res.round(4).to_string(index=False))
stated = bins_res.iloc[[2, 3, 4, 7]]
print("\nRolling accuracy: saline higher at p < 0.05 in 10-15, 15-20, 20-25, 35-40:",
      bool((stated.rolling_saline_higher & (stated.rolling_p < 0.05)).all()),
      "; p < 0.01 in the first three:", bool((bins_res.iloc[[2, 3, 4]].rolling_p < 0.01).all()),
      "; 30-35 not significant:", bool(bins_res.iloc[6].rolling_p >= 0.05))
print("Mice without odor trials in a bin are absent from it (counts above).")
grand = binned.groupby(["condition", "bin"])["rolling"].agg(["mean", "std"]).unstack(0)
fig, ax = plt.subplots(figsize=(5, 3))
for cnd, c in (("DOI", "purple"), ("saline", "k")):
    ax.errorbar(grand.index * 5 + 2.5, grand[("mean", cnd)], yerr=grand[("std", cnd)], color=c, label=cnd, capsize=3)
ax.set(xlabel="time in session (min)", ylabel="mean rolling accuracy (%)")
ax.legend()
plt.show()
print("Plotted means range %.1f to %.1f %%; error bars are 1 SD." % (grand.xs("mean", axis=1, level=0).min().min(), grand.xs("mean", axis=1, level=0).max().max()))

**Result: reproduced.** Saline rolling accuracy is higher than DOI in every bin. It is higher at
p < 0.01 in the 10-15, 15-20 and 20-25 min bins (p = 0.0076, 0.0037, 0.0070) and at p = 0.039 in
the 35-40 min bin, the paper's value, and the 30-35 min bin is not significant (p = 0.052). The
25-30 min bin, which the text does not state, has p = 0.018, and the first two bins are not
significant. The per-bin fraction correct (variant) is printed alongside.

### fig1d-fewer-trials

> Mice complete fewer trials in the 40 min DOI session than in the saline session.

In [ ]:
res = pd.DataFrame([paired("trials_odor", label="(a) odor trials, the code"),
                    paired("trials_all", label="(b) all trials, the Methods")])
print(res.round(4).to_string(index=False))
print("\nPer mouse, all trials:")
print(wide["trials_all"].assign(sex=sex_of).T.to_string())

**Result: reproduced.** DOI sessions have fewer trials under both readings: odor trials median
difference -9.5, p = 0.011 (the paper's 0.011); all trials -8, p = 0.018.

### fig1d-fewer-trials-females-only

> The drop in trials completed under DOI is driven by females; female mice perform fewer trials on
> DOI than saline, while male mice show no significant difference.

In [ ]:
F = [m for m in wide.index if sex_of[m] == "F"]
M = [m for m in wide.index if sex_of[m] == "M"]
res = pd.DataFrame([paired(c, mice, f"{lab}, {sx}") for c, lab in (("trials_odor", "(a) odor trials"), ("trials_all", "(b) all trials"))
                    for sx, mice in (("females", F), ("males", M))])
print(res.round(4).to_string(index=False))

**Result: reproduced.** Under both readings females (12) do fewer trials on DOI (odor trials p =
0.010, the paper's 0.011; all trials p = 0.015) and males (10) do not differ (p = 0.39, the
paper's 0.39; all trials p = 0.44).

### fig1e-repoke-no-difference

> Mice poke into the initiation port during the ITI at the same rate in the DOI and saline
> conditions, showing an equivalent motivation to do the task.

In [ ]:
res = pd.DataFrame([paired("repoke_rate", label="repokes per ITI, later cohort")])
print(res.round(4).to_string(index=False))
print("\nPer mouse:")
print(wide["repoke_rate"].dropna().round(3).T.to_string())

**Result: reproduced.** Over the 17 later-cohort mice the repoke rate does not differ (median
difference -0.10 pokes per ITI, p = 0.38; the paper's p = 0.27).

### iti-false-report-no-difference

> Mice rarely enter a reward port during the ITI after a trial-initiation attempt, and DOI does
> not make them do so more than saline.

In [ ]:
lat = per_session[per_session.cohort == "later"]
rows = []
for col, lab in (("false_reports", "(a) false reports per session"), ("false_report_rate", "(b) per ITI")):
    a, b = lat[lat.condition == "DOI"][col], lat[lat.condition == "saline"][col]
    w, p = ds.r_ranksum(a, b)
    rows.append({"reading": lab, "n_DOI": len(a), "n_saline": len(b), "DOI_mean": a.mean(), "saline_mean": b.mean(),
                 "DOI_median": a.median(), "saline_median": b.median(), "W": w, "p": p})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nFalse reports per session (counts):")
print(lat.pivot(index="mouse", columns="condition", values="false_reports").T.to_string())
print("Sessions with any false report: DOI", int((lat[lat.condition == 'DOI'].false_reports > 0).sum()),
      ", saline", int((lat[lat.condition == 'saline'].false_reports > 0).sum()), "of 17 each;",
      "false reports over all later-cohort ITIs:", int(lat.false_reports.sum()), "of", int((lat.trials_all - 1).sum()))
print("The Results' variant (reward-port entries in the ITI) cannot be run: the dandiset has no reward-port events.")

**Result: reproduced.** The `false_alarm` column of the trial table, the Methods' false odor
report, gives 3 false reports in 2 of 17 DOI sessions and 3 in 2 of 17 saline sessions, 6 of
3227 ITIs; the rank-sum test gives p = 1 under both readings, the paper's value. The Results'
reward-port variant could not be run.

## Figure 2: reward-port duration (blocked)

The nine claims of Figure 2 and its supplement need the times of entering and leaving the reward
port after each trial. The files hold no such events: the trial table has start, stop, side,
outcome, set ITI and, for the later cohort, false-alarm and repoke counts; the only event series is
the HTR series; `acquisition`, `stimulus` and `analysis` are empty (printed below). The code reads
the durations from pre-computed files that are not in its repository either.

In [ ]:
import h5py, remfile
path, url = ds.asset_urls()[20]
with h5py.File(remfile.File(url), "r") as f:
    groups = {}
    f.visit(lambda name: groups.setdefault(name.split("/")[0], []).append(name))
    print(path)
    for g in ("acquisition", "analysis", "stimulus", "intervals", "processing"):
        members = [n for n in groups.get(g, []) if n.count("/") <= 2]
        print(f"  {g}: {members[:12]}")
    print("  trial columns:", list(f["intervals/trials"].attrs["colnames"]))

## Figure 3: trial and ITI duration

Trial duration is end minus start; ITI duration is the next trial's start minus this trial's end;
per mouse the mean (Methods, Figure 3; fig3_cleaned.Rmd).

### fig3a-odor-trial-duration-longer

> Mice take longer to complete odor trials in the DOI condition than in the saline condition.

In [ ]:
res = pd.DataFrame([paired("dur_odor", label="odor-trial duration (s)")])
print(res.round(6).to_string(index=False))
print("\nPer mouse (s):")
print(wide["dur_odor"].round(2).T.to_string())

**Result: reproduced.** Odor trials last longer on DOI (median difference 0.26 s, longer in 21 of
22 mice), p = 5e-06 against the paper's p < 0.001.

### fig3a-omission-duration-no-difference

> The duration of omission trials does not differ between DOI and saline.

In [ ]:
res = pd.DataFrame([paired("dur_omission", label="omission-trial duration (s)")])
print(res.round(4).to_string(index=False))

**Result: reproduced.** Omission-trial duration does not differ (median difference 0.55 s, p =
0.32; the paper's p = 0.26).

### fig3b-iti-duration-no-difference

> ITI duration does not differ between DOI and saline.

In [ ]:
res = pd.DataFrame([paired("iti", label="mean ITI (s)")])
print(res.round(4).to_string(index=False))
print("\nPer mouse (s):")
print(wide["iti"].round(2).T.to_string())

**Result: reproduced.** The mean ITI does not differ at p < 0.05 (median difference 1.4 s, longer on
DOI, p = 0.098; the paper's p = 0.19).

## Figure 3C, 3D and supplement: speed

Nose speed as the claims file gives it (cleaning of `ds.clean_nose`, frame-to-frame displacement
x 80 frames/s x 0.0237 cm/px, frames above 160 cm/s dropped), per mouse the mean over the frames
of each period. Periods by the code's clock (frame i at i / 80 s), and as a variant by the clock
fitted in the data check.

In [ ]:
rows = []
for s in sessions:
    x, y, _ = ds.clean_nose(s)
    sp = ds.speed_cm_s(x, y)
    keep = sp <= ds.MAX_SPEED_CM_S
    fps, off = CLOCK[(s["mouse"], s["condition"])]
    for clk, times in (("code", ds.frame_times(len(x))), ("fitted", ds.frame_times(len(x), fps) - off)):
        lab = ds.label_periods(times, s["trials"])
        rows.append({"mouse": s["mouse"], "condition": s["condition"], "clock": clk,
                     "within": sp[keep & (lab > 0)].mean(), "iti": sp[keep & (lab == 0)].mean(),
                     "odor": sp[keep & (lab == 1)].mean(), "omission": sp[keep & (lab == 3)].mean(),
                     "frames_dropped_over_160": int((~keep[1:]).sum()), "frames": len(sp),
                     "within_frames_frac": float((lab > 0).mean())})
speed = pd.DataFrame(rows)
print("Frames dropped above 160 cm/s, all sessions:", int(speed[speed.clock == "code"].frames_dropped_over_160.sum()),
      "of", int(speed[speed.clock == "code"].frames.sum()))
print("Fraction of frames within trials: code clock median %.3f, fitted %.3f"
      % (speed[speed.clock == "code"].within_frames_frac.median(), speed[speed.clock == "fitted"].within_frames_frac.median()))
print("Per-mouse mean speeds (cm/s), range over mice:")
print(speed.groupby(["clock", "condition"])[["within", "iti", "odor", "omission"]].agg(["min", "median", "max"]).round(2).to_string())
SW = {clk: g.drop(columns="clock").pivot(index="mouse", columns="condition") for clk, g in speed.groupby("clock")}
fitted_ok = set(clock[clock.fit_ok].mouse.groupby(clock[clock.fit_ok].mouse).size().loc[lambda v: v == 2].index)
print("Mice whose two sessions both have a good clock fit:", len(fitted_ok))


def speed_test(a, b, clk="code", mice=None):
    w = SW[clk] if mice is None else SW[clk].loc[sorted(mice)]
    xa, xb = w[a], w[b]
    v, p = ds.r_signrank(xa, xb)
    wr, pr = ds.r_ranksum(xa, xb)
    d = xa - xb
    return {"clock": clk if mice is None else f"{clk}, {len(mice)} mice", "n": len(d), "A_median": xa.median(), "B_median": xb.median(),
            "median_A_minus_B": d.median(), "A_lower_in": int((d < 0).sum()), "signrank_p": p, "ranksum_p": pr}


def speed_block(a, b):
    out = pd.DataFrame([speed_test(a, b, "code"), speed_test(a, b, "fitted"), speed_test(a, b, "fitted", fitted_ok)])
    print(f"A = {a}, B = {b}")
    print(out.round(4).to_string(index=False))

### fig3c-speed-within-trial-lower

> DOI lowers movement speed within trials, odor and omission trials pooled, compared with saline.

In [ ]:
speed_block(("within", "DOI"), ("within", "saline"))
print("\nBy sex, code clock:")
for sx in "FM":
    print(sx, {k: round(v, 4) if isinstance(v, float) else v for k, v in
               speed_test(("within", "DOI"), ("within", "saline"), "code", [m for m in SW["code"].index if sex_of[m] == sx]).items()})
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
for i, (per, title) in enumerate((("within", "within trial"), ("iti", "ITI"))):
    w = SW["code"]
    ax[i].scatter(w[(per, "saline")], w[(per, "DOI")])
    lim = [0, max(w[per].max().max(), 1) * 1.1]
    ax[i].plot(lim, lim, "k:")
    ax[i].set(xlabel="saline speed (cm/s)", ylabel="DOI speed (cm/s)", title=title + " (code clock)", xlim=lim, ylim=lim)
plt.show()
print("Plotted per-mouse speeds span %.2f to %.2f cm/s; none outside the axes."
      % (SW["code"][["within", "iti"]].min().min(), SW["code"][["within", "iti"]].max().max()))

**Result: partially reproduced.** With the claims file's method (code clock, signed-rank test)
within-trial speed is lower on DOI in 16 of 22 mice (median difference -0.63 cm/s) but p = 0.063,
against the paper's 0.011; the rank-sum test the Results name gives p = 0.51. With the fitted
clock, which aligns the frames with the trials, DOI and saline do not differ (p = 0.82; 0.89 in
the 13 mice whose two fits are good).

### fig3d-speed-iti-lower

> DOI lowers movement speed during the ITI compared with saline.

In [ ]:
speed_block(("iti", "DOI"), ("iti", "saline"))

**Result: partially reproduced.** ITI speed is lower on DOI in 15 of 22 mice (median difference
-0.60 cm/s) but p = 0.079 with the claims file's method, against the paper's 0.036; with the
fitted clock p = 0.063 (0.057 in the 13 mice with good fits). The rank-sum variant gives p = 0.57.

### fig3supp-odor-trial-speed-no-difference

> Within odor trials alone, speed does not differ between DOI and saline sessions.

In [ ]:
speed_block(("odor", "DOI"), ("odor", "saline"))

**Result: reproduced.** Odor-trial speed does not differ (signed-rank p = 0.19, median difference
-0.54 cm/s; rank-sum p = 0.70); with the fitted clock p = 0.68.

### fig3supp-doi-iti-slower-than-odor

> Within the DOI condition, mice move more slowly in the ITI than in odor trials.

In [ ]:
speed_block(("iti", "DOI"), ("odor", "DOI"))

**Result: reproduced.** On DOI the ITI is slower than odor trials in 19 of 22 mice (median
difference -0.83 cm/s, p = 0.0002, the paper's p < 0.001); with the fitted clock the difference is
larger (-3.5 cm/s, p < 0.0001).

### fig3supp-doi-omission-no-difference

> Within the DOI condition, speed does not differ between odor and omission trials, nor between
> the ITI and omission trials.

In [ ]:
speed_block(("odor", "DOI"), ("omission", "DOI"))
print()
speed_block(("iti", "DOI"), ("omission", "DOI"))

**Result: partially reproduced.** With the claims file's method odor and omission trials do not
differ (p = 0.31) but the ITI is slower than omission trials (median difference -0.70 cm/s, p =
0.0059, against the paper's 0.068): one of the two comparisons is significant. The rank-sum
variants the Results name give p = 0.62 and 0.37. With the fitted clock both comparisons are
significant (p = 0.0003 and 0.0001).

### fig3supp-saline-iti-slower

> In saline sessions, mice move more slowly in the ITI than in odor trials and in omission trials.

In [ ]:
speed_block(("iti", "saline"), ("odor", "saline"))
print()
speed_block(("iti", "saline"), ("omission", "saline"))

**Result: reproduced.** On saline the ITI is slower than odor trials (median difference -0.43
cm/s, p = 0.025, the paper's 0.002) and than omission trials (-0.98 cm/s, p = 0.016, the paper's
0.017); with the fitted clock both at p < 0.0001.

### fig3supp-saline-odor-omission-no-difference

> In saline sessions, speed does not differ between odor and omission trials.

In [ ]:
speed_block(("odor", "saline"), ("omission", "saline"))

**Result: reproduced.** On saline odor and omission trials do not differ (p = 0.68, median
difference -0.18 cm/s; the paper's 0.54); with the fitted clock p = 0.32.

## Figure 4: sniff frequency

Sniff frequency is 1000 over the interval in ms between successive inhalations (`ds.sniff_table`),
in the 14 DOI and 13 saline sessions the code keeps. The claims file names two readings of the
Methods' exclusions, (a) the 5th to 95th percentile of durations per session and (b) over the
sessions of a condition, each with durations outside 10 to 1000 ms dropped. The file's cycles are
the same detections the code reads, so the code's computation (no further exclusion) is shown as a
third reading.

In [ ]:
sniffs = pd.concat([ds.sniff_table(s) for s in sessions
                    if s["sniff_start"] is not None and (s["mouse"], s["session"]) not in ds.FAULTY_SNIFF])
READ = ds.sniff_readings(sniffs)
for k, d in READ.items():
    print(f"{k}: sniffs DOI {int((d.condition == 'DOI').sum())}, saline {int((d.condition == 'saline').sum())}; "
          f"sessions DOI {d[d.condition == 'DOI'].mouse.nunique()}, saline {d[d.condition == 'saline'].mouse.nunique()}; "
          f"frequency {d.frequency.min():.2f} to {d.frequency.max():.2f} Hz; duration {d.duration_ms.min():.0f} to {d.duration_ms.max():.0f} ms")

### fig4abc-sniff-frequency-higher

> Pooling all sessions with a usable sniff signal, sniff frequency is higher in DOI sessions than
> in saline sessions.

In [ ]:
rng_seed = 4
rows = []
for k, d in READ.items():
    a = d.frequency[d.condition == "DOI"].to_numpy()
    b = d.frequency[d.condition == "saline"].to_numpy()
    z, p = ds.fisher_pitman(a, b, n_resample=10000, seed=rng_seed)
    rows.append({"reading": k, "n_DOI": len(a), "n_saline": len(b), "DOI_mean_Hz": a.mean(), "saline_mean_Hz": b.mean(),
                 "Z": z, "p_10000_resamples": p, "p_normal": 2 * stats.norm.sf(abs(z))})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Sessions with a usable signal: DOI", READ["a_session"][READ["a_session"].condition == "DOI"].mouse.nunique(),
      "(expected 14), saline", READ["a_session"][READ["a_session"].condition == "saline"].mouse.nunique(), "(expected 13)")
d = READ["a_session"]
fig, ax = plt.subplots(figsize=(5, 3))
bins = np.arange(0, 16.25, 0.25)
for cnd, c in (("DOI", "purple"), ("saline", "grey")):
    ax.hist(d.frequency[d.condition == cnd], bins=bins, density=True, histtype="step", color=c, label=cnd)
ax.set(xlabel="sniff frequency (Hz)", ylabel="density")
ax.legend()
plt.show()
print("Sniffs above the 16 Hz axis, reading (a):", int((d.frequency > 16).sum()), "of", len(d))

**Result: reproduced.** The sessions are 14 DOI and 13 saline, the expected counts. DOI sniff
frequency is higher under every reading (mean 7.78 against 7.55 Hz under reading a), Z = 25.2
(a), 28.4 (b) and 19.5 (the code's), against the paper's Z = 24.9, with no permutation of 10000 as
extreme under any reading (p < 0.0001).

### fig4supp-ks-every-mouse

> Every mouse with matched DOI and saline sniff sessions has a sniff-frequency distribution
> shifted higher in the DOI condition.

In [ ]:
rows = []
for k, d in READ.items():
    for m, g in d.groupby("mouse"):
        if g.condition.nunique() < 2:
            continue
        r = stats.ks_2samp(g.frequency[g.condition == "DOI"], g.frequency[g.condition == "saline"], alternative="less")
        rows.append({"reading": k, "mouse": m, "D": r.statistic, "p": r.pvalue,
                     "DOI_median": g.frequency[g.condition == "DOI"].median(), "saline_median": g.frequency[g.condition == "saline"].median()})
ks = pd.DataFrame(rows)
print(ks.pivot(index="mouse", columns="reading", values="p").map(lambda v: f"{v:.2g}").to_string())
print()
print(ks.groupby("reading").agg(matched_mice=("mouse", "size"), p_below_0_001=("p", lambda v: int((v < 0.001).sum())),
                                p_below_0_05=("p", lambda v: int((v < 0.05).sum()))).to_string())
print("\nMedian frequency (Hz) per mouse, reading (a):")
print(ks[ks.reading == "a_session"][["mouse", "DOI_median", "saline_median"]].round(2).to_string(index=False))

**Result: reproduced.** 13 mice have matched sessions, the expected number. Under reading (b) all
13 have a one-sided KS p < 0.001; under reading (a) and the code's computation 11 of 13 do, and the
other two (2209 and 2211) have p of 0.0031 to 0.011; all 13 have p < 0.05 under every reading. The
criterion names both readings, and the claim holds under (b); under (a) it would be partially
reproduced. Per-mouse medians are higher on DOI in 8 of the 13 mice.

### fig4d-sniff-frequency-by-period

> Mice sniff faster in the DOI condition in odor trials, in omission trials and during the ITI.

In [ ]:
rows = []
for k, d in READ.items():
    for per, lab in ((1, "odor trials"), (3, "omission trials"), (0, "ITI")):
        a = d.frequency[(d.condition == "DOI") & (d.period == per)].to_numpy()
        b = d.frequency[(d.condition == "saline") & (d.period == per)].to_numpy()
        z, p = ds.fisher_pitman(a, b, n_resample=10000 if k == "a_session" else 0, seed=rng_seed)
        rows.append({"reading": k, "period": lab, "n_DOI": len(a), "n_saline": len(b), "DOI_mean": a.mean(),
                     "saline_mean": b.mean(), "Z": z, "p_10000_resamples": p if k == "a_session" else np.nan,
                     "p_normal": 2 * stats.norm.sf(abs(z))})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Resampled p is computed for reading (a); the others give the normal approximation of the same statistic.")

**Result: reproduced.** DOI sniffing is faster in all three periods under every reading (Z
printed above; the paper's 3.38, 2.71 and 18.4), each at p < 0.05. Sniffs are placed in periods
by the trial table's clock, which the data check found consistent with the sniff clock.

### fig4e-median-sniff-no-difference

> Per-mouse median sniff frequency does not differ between DOI and saline.

In [ ]:
rows = []
for k, d in READ.items():
    med = d.groupby(["mouse", "condition"]).frequency.median().unstack()
    both = med.dropna()
    v, p = ds.r_signrank(both.DOI, both.saline)
    rows.append({"reading": k, "mice_with_both": len(both), "median_DOI_minus_saline": (both.DOI - both.saline).median(),
                 "DOI_higher_in": int((both.DOI > both.saline).sum()), "V": v, "p": p})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** Over the 13 mice with both sessions, the per-mouse median frequency does
not differ at p < 0.05 (p = 0.11 under reading a and the code's, the paper's 0.11; 0.15 under b);
it is higher on DOI in 8 of 13 mice (median difference 0.53 Hz under a).

## Figure 5: occupancy and head twitches

Frames as fig5_cleaned.Rmd builds them (`ds.map_frames`): nose position after cleaning, the first
cohort's x flipped (1200 - x), negative positions set to 0, periods by the code's clock (and the
fitted clock as a variant), HTRs on the frame nearest each timestamp.

In [ ]:
FR = {}
for s in sessions:
    FR[(s["mouse"], s["condition"], "code")] = ds.map_frames(s)
allx = np.concatenate([f.x.to_numpy()[::10] for (m, c, r), f in FR.items()])
ally = np.concatenate([f.y.to_numpy()[::10] for (m, c, r), f in FR.items()])
BOUNDS = (*np.percentile(allx, [0.5, 99.5]), *np.percentile(ally, [0.5, 99.5]))
print("Nose positions pooled over sessions (px): x %.0f to %.0f, y %.0f to %.0f; 0.5 to 99.5 percentile box %s"
      % (allx.min(), allx.max(), ally.min(), ally.max(), np.round(BOUNDS, 0)))
print("Arena from that box: %.1f x %.1f cm at 0.0237 cm/px" % ((BOUNDS[1] - BOUNDS[0]) * ds.CM_PER_PX, (BOUNDS[3] - BOUNDS[2]) * ds.CM_PER_PX))

### fig5ab-occupancy-similar

> Mice occupy the arena similarly in the DOI and saline conditions; DOI does not make them spend
> more time in task-irrelevant areas such as the corners or spread more evenly.

In [ ]:
rows = []
for (m, c, r), f in FR.items():
    rows.append({"mouse": m, "condition": c, **ds.occupancy_measures(f, BOUNDS)})
occ = pd.DataFrame(rows).pivot(index="mouse", columns="condition")
out = []
for k in ("entropy", "corners_5cm", "corners_10cm"):
    v, p = ds.r_signrank(occ[(k, "DOI")], occ[(k, "saline")])
    out.append({"measure": k, "DOI_median": occ[(k, "DOI")].median(), "saline_median": occ[(k, "saline")].median(),
                "median_DOI_minus_saline": (occ[(k, "DOI")] - occ[(k, "saline")]).median(), "p": p})
print(pd.DataFrame(out).round(4).to_string(index=False))


def occ_map(cond):
    maps = []
    for (m, c, r), f in FR.items():
        if c != cond:
            continue
        g = f[f.speed <= ds.MAX_SPEED_CM_S]
        H, _, _ = np.histogram2d(g.x, g.y, bins=[np.arange(0, 1249, ds.OCC_BIN_PX), np.arange(0, 721, ds.OCC_BIN_PX)])
        maps.append(H / H.sum())
    return np.mean(maps, axis=0)
mD, mS = occ_map("DOI"), occ_map("saline")
print("Pearson r between the grand-mean maps (22 mice each): %.3f" % np.corrcoef(mD.ravel(), mS.ravel())[0, 1])
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
vmax = max(mD.max(), mS.max())
for a, mp, t in ((ax[0], mS, "saline"), (ax[1], mD, "DOI")):
    im = a.imshow(mp.T, origin="upper", vmin=0, vmax=vmax, extent=(0, 1248, 720, 0))
    a.set(title=f"occupancy, {t}", xlabel="x (px, odor end right)", ylabel="y (px)")
fig.colorbar(im, ax=ax, label="fraction of frames")
plt.show()
print("Map values range 0 to %.4f; the colour scale covers it." % vmax)

**Result: reproduced.** No measure differs at p < 0.05: occupancy entropy (median difference
0.11 bits, p = 0.063), the fraction of frames in 5 cm corners (p = 0.25) and in 10 cm corners (p =
0.70). The grand-mean maps correlate with r = 0.919. With the arena about 27.5 by 15.6 cm at the
code's scale, 10 cm corner squares hold 85% of the frames, so that reading says little.

### htr-event-link (added here)

> The dataset's `task_event_timestamps` are the head-twitch (HTR) times the paper analysed.

The files do not name the series. The conversion script loads it from the folders "Time Stamps
Task" and "Time Stamps Task Logan", which fig5_cleaned.Rmd reads as HTR timestamps, by the same
mouse and session names. HTRs are induced by DOI, so the series should be present in all 22 DOI
sessions (the paper's n = 22) and much more frequent there than in saline sessions.

In [ ]:
ev_counts = info.pivot(index="mouse", columns="condition", values="events")
print("DOI sessions with the series:", int((ev_counts.DOI > 0).sum()), "of 22; saline sessions:", int((ev_counts.saline > 0).sum()), "of 22")
both = ev_counts[(ev_counts.DOI > 0) & (ev_counts.saline > 0)]
print("Mice with the series in both sessions:", len(both), "; DOI count higher in", int((both.DOI > both.saline).sum()))
print("Events per minute of the 40 min session, DOI: median %.2f (range %.2f to %.2f); saline (where present): median %.2f"
      % ((ev_counts.DOI / 40).median(), (ev_counts.DOI / 40).min(), (ev_counts.DOI / 40).max(), (ev_counts.saline[ev_counts.saline > 0] / 40).median()))
print("Source folder by cohort:", info[info.events > 0].groupby(["cohort", "event_source"]).size().to_dict())
print(ev_counts.T.to_string())

**Result: reproduced.** The series is in all 22 DOI sessions, as the paper's n = 22 needs, and in
10 saline sessions; in each of the 10 mice with both, the DOI session has more events (median 3.67
per minute on DOI, 0.47 on saline where present). Together with the conversion script's sources
this links the series to the HTR times. The DOI sessions of 2197 and 2204 have only 30 and 24
events. The files themselves do not name the series.

In [ ]:
HTR_READINGS = ("code", "file", "frames")
htr_tables = {}
for reading in HTR_READINGS:
    for clk in ("code", "fitted"):
        if clk == "fitted" and reading != "code":
            continue
        for s in sessions:
            if s["condition"] != "DOI":
                continue
            fps, off = (ds.CODE_FPS, 0.0) if clk == "code" else CLOCK[(s["mouse"], "DOI")]
            f = FR[(s["mouse"], "DOI", "code")] if (reading, clk) == ("code", "code") else ds.map_frames(s, fps, off, reading)
            for period in ("all", "within", "iti"):
                htr_tables.setdefault((reading, clk, period), []).append(ds.htr_rate_table(f, s["mouse"], period))
htr_tables = {k: pd.concat(v) for k, v in htr_tables.items()}


def htr_perm(period):
    rows = []
    for (reading, clk, per), T in htr_tables.items():
        if per != period:
            continue
        rx, px = ds.perm_corr_test(T.rate, T.bx)
        ry, py = ds.perm_corr_test(T.rate, T.by)
        rows.append({"timestamps": reading, "clock": clk, "rows": len(T), "mice": T.mouse.nunique(),
                     "HTRs_counted": int(T.htr_count.sum()), "r_x": rx, "p_x": px, "r_y": ry, "p_y": py})
    return pd.DataFrame(rows)

### fig5supp-htr-within-trial-x-not-y

> During trials, HTR rate (normalized to occupancy) varies with the X coordinate of the arena but
> not with the Y coordinate.

In [ ]:
print(htr_perm("within").round(4).to_string(index=False))
T = htr_tables[("code", "code", "within")]
print("\nWithin-trial rate (HTR/min) per bin, code reading: range %.2f to %.2f; bins with rate 0: %d of %d"
      % (T.rate.min(), T.rate.max(), int((T.rate == 0).sum()), len(T)))

**Result: reproduced.** With the claims file's method (code clock, code timestamps) the
within-trial HTR rate correlates with x (r = 0.056, permutation p = 0.0048; the paper's 0.002) and
not with y (p = 0.26; the paper's 0.92); the other two timestamp readings agree (x p = 0.0016 and
0.0020, y p = 0.92 and 0.86). With the fitted clock only 151 of 3057 HTRs fall within trials and
the x correlation vanishes (r = 0.001, p = 0.96; y p = 0.065), so the result depends on the
code's clock.

### fig5e-htr-combined-x

> With trial and ITI periods combined, mice head-twitch more often closer to the odor end of the
> arena, as the X coordinate increases.

In [ ]:
print(htr_perm("all").round(4).to_string(index=False))
T = htr_tables[("code", "code", "all")]
print("\nMice in the map:", T.mouse.nunique(), "(expected 22)")
grand = T.groupby(["bx", "by"]).rate.mean().unstack()
print("Grand-mean rate by x bin (mean over y bins, HTR/min):")
print(grand.mean(axis=1).round(2).to_string())
fig, ax = plt.subplots(figsize=(5, 3))
im = ax.imshow(grand.T.to_numpy(), origin="upper", extent=(grand.index.min(), grand.index.max() + 100, grand.columns.max() + 100, grand.columns.min()))
ax.set(xlabel="x (px, odor end right)", ylabel="y (px)", title="HTR rate, DOI (grand mean)")
fig.colorbar(im, label="HTR/min")
plt.show()
print("Map range %.2f to %.2f HTR/min." % (np.nanmin(grand.to_numpy()), np.nanmax(grand.to_numpy())))

**Result: reproduced.** Over the 22 DOI mice (the expected 22), with trial and ITI frames
combined, the HTR rate rises with x toward the odor and reward ports (r = 0.149, none of 10000
permutations as extreme, p < 0.0001), under every timestamp reading; the grand-mean rate rises
from 1.97 HTR/min in the first x bin to about 6.5 at the odor end. This does not depend on the
clock.

### fig5supp-htr-iti-x

> During the ITI, mice head-twitch more often toward the end of the arena past the decision line
> (as X increases), but not specifically next to the reward ports on the Y coordinate.

In [ ]:
print(htr_perm("iti").round(4).to_string(index=False))

**Result: reproduced.** In the ITI the HTR rate rises with x (r = 0.152, p < 0.0001) and does not
vary with y (p = 0.55 under the code's reading; 0.08 and 0.25 under the other timestamp readings,
0.13 with the fitted clock).

### fig5f-htr-rate-trial-vs-iti

> HTR rate does not differ between time within trials and time in the ITI.

In [ ]:
rows = []
for reading in HTR_READINGS:
    for clk in ("code", "fitted"):
        for s in sessions:
            if s["condition"] != "DOI":
                continue
            h = ds.htr_times(s, reading)
            if clk == "fitted":
                fps, off = CLOCK[(s["mouse"], "DOI")]
                h = h * ds.CODE_FPS / fps - off      # video time to trial time
            t = s["trials"]
            within = np.zeros(len(h), bool)
            for st, en in zip(t.start_time, t.stop_time):
                within |= (h >= st) & (h <= en)
            trial_time = (t.stop_time - t.start_time).sum()
            rows.append({"timestamps": reading, "clock": clk, "mouse": s["mouse"], "sex": s["sex"],
                         "htr": len(h), "in_trial": int(within.sum()),
                         "rate_trial": within.sum() / trial_time * 60, "rate_iti": (~within).sum() / (ds.SESSION_S - trial_time) * 60})
htr_rates = pd.DataFrame(rows)
out = []
for (reading, clk), g in htr_rates.groupby(["timestamps", "clock"]):
    v, p = ds.r_signrank(g.rate_trial, g.rate_iti)
    out.append({"timestamps": reading, "clock": clk, "mice": len(g), "trial_median": g.rate_trial.median(), "iti_median": g.rate_iti.median(),
                "median_trial_minus_iti": (g.rate_trial - g.rate_iti).median(), "V": v, "p": p})
print(pd.DataFrame(out).round(4).to_string(index=False))
print("\nBy cohort, code timestamps:")
for (clk, coh), g in htr_rates[htr_rates.timestamps == "code"].assign(cohort=lambda d: np.where(d.mouse.isin(ds.FIRST_COHORT), "first", "later")).groupby(["clock", "cohort"]):
    v, p = ds.r_signrank(g.rate_trial, g.rate_iti)
    print(f"  clock {clk}, {coh} cohort: n={len(g)}, trial median {g.rate_trial.median():.2f}, ITI median {g.rate_iti.median():.2f} HTR/min, p={p:.4f}")
good = htr_rates[(htr_rates.timestamps == "code") & (htr_rates.clock == "fitted") & htr_rates.mouse.isin(clock[clock.fit_ok & (clock.condition == "DOI")].mouse)]
v, p = ds.r_signrank(good.rate_trial, good.rate_iti)
print(f"  clock fitted, DOI sessions with a good fit: n={len(good)}, trial median {good.rate_trial.median():.2f}, ITI median {good.rate_iti.median():.2f}, p={p:.4f}")
print("\nPer mouse, code reading and clock (HTR/min):")
print(htr_rates[(htr_rates.timestamps == "code") & (htr_rates.clock == "code")][["mouse", "htr", "in_trial", "rate_trial", "rate_iti"]].round(2).to_string(index=False))

**Result: reproduced.** With the code's method the HTR rate within trials and in the ITI does not
differ (medians 3.67 and 3.68 HTR/min, signed-rank p = 0.50; 0.52 and 0.70 under the other
timestamp readings; the paper's p = 1). With the fitted clock, HTRs are about a third as frequent
within trials as in the ITI (medians 1.38 and 3.88 HTR/min, p < 0.0001, under every timestamp
reading), the opposite of the claim: the claim holds only with the code's clock, which the data
check found does not align the two clocks.

### fig5supp-htr-no-sex-difference

> HTR rate over the 40 min DOI session does not differ between male and female mice.

In [ ]:
g = htr_rates[(htr_rates.timestamps == "code") & (htr_rates.clock == "code")].assign(rate=lambda d: d.htr / 40)
w, p = ds.r_ranksum(g.rate[g.sex == "F"], g.rate[g.sex == "M"], exact=False)
print("HTR per min over 40 min: females n=%d median %.2f mean %.2f; males n=%d median %.2f mean %.2f; rank-sum W=%.1f p=%.4f"
      % ((g.sex == "F").sum(), g.rate[g.sex == "F"].median(), g.rate[g.sex == "F"].mean(),
         (g.sex == "M").sum(), g.rate[g.sex == "M"].median(), g.rate[g.sex == "M"].mean(), w, p))

**Result: reproduced.** HTR rate over the session does not differ between the sexes (females
median 3.61, males 3.99 HTR/min; rank-sum with the normal approximation, p = 0.31, the paper's
0.31). It needs no clock.

## Summary

| Claim | Outcome |
|---|---|
| fig1b-accuracy-lower-doi | reproduced |
| fig1c-accuracy-time-bins | reproduced |
| fig1d-fewer-trials | reproduced |
| fig1d-fewer-trials-females-only | reproduced |
| fig1e-repoke-no-difference | reproduced |
| iti-false-report-no-difference | reproduced |
| fig2a to fig2d-supp (9 claims) | blocked, data missing |
| fig3a-odor-trial-duration-longer | reproduced |
| fig3a-omission-duration-no-difference | reproduced |
| fig3b-iti-duration-no-difference | reproduced |
| fig3c-speed-within-trial-lower | partially reproduced |
| fig3d-speed-iti-lower | partially reproduced |
| fig3supp-odor-trial-speed-no-difference | reproduced |
| fig3supp-doi-iti-slower-than-odor | reproduced |
| fig3supp-doi-omission-no-difference | partially reproduced |
| fig3supp-saline-iti-slower | reproduced |
| fig3supp-saline-odor-omission-no-difference | reproduced |
| fig4abc-sniff-frequency-higher | reproduced |
| fig4supp-ks-every-mouse | reproduced (under reading b) |
| fig4d-sniff-frequency-by-period | reproduced |
| fig4e-median-sniff-no-difference | reproduced |
| fig5ab-occupancy-similar | reproduced |
| htr-event-link | reproduced |
| fig5supp-htr-within-trial-x-not-y | reproduced |
| fig5e-htr-combined-x | reproduced |
| fig5supp-htr-iti-x | reproduced |
| fig5f-htr-rate-trial-vs-iti | reproduced |
| fig5supp-htr-no-sex-difference | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Statistics are computed in Python with R's
defaults reproduced; the permutation tests draw other random permutations than R's. The
Fisher-Pitman p-values of the per-period tests are resampled for reading (a) only. The occupancy
measures and the arena box are this attempt's readings of the paper's prediction. Frames are split
into trial periods by the code's 80 frames/s clock, with a fitted clock as a variant.

**What was found about the dataset.** The reward-port entries and exits of Figure 2 are not in
it. The tracking has no frame times; its stated rate is an estimate, and neither it nor the
code's 80 frames/s aligns the frames with the trial table, while a fitted clock (about 80.5
frames/s in the later cohort, 72 to 75 in the first) does. The claims that split frames or HTRs
by trial period follow the code's clock as the claims file fixes it; with the fitted clock the
within-trial speed difference and the within-trial HTR x preference vanish, and HTRs are rarer
within trials than in the ITI (fig5f). The HTR series is not named in the files, and its
later-cohort timestamps store a frame count as hundredths of a second. The files do not say which
side code is left (the notes show side 2 is left). The saline session of mouse 2182 gives the
wrong sex.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")